In [0]:
# Live 3 (C) — Consulta (SELECT / WHERE / ORDER BY)
# Objetivo: responder perguntas comuns com queries legíveis e fáceis de validar.
# Formato: células intercalando SQL (spark.sql) e PySpark (DataFrame API).

from pyspark.sql import functions as F
from pyspark.sql import types as T

def d(df, n=20):
    """Helper de visualização: usa display() no Databricks; senão, df.show()."""
    try:
        display(df)
    except NameError:
        df.show(n, truncate=False)

def sanity_counts(df_base, df_filtered, label_base="BASE", label_filtered="RECORTE"):
    """
    Validação rápida padrão:
      - contagem antes e depois do WHERE
    """
    base_n = df_base.count()
    filt_n = df_filtered.count()
    print(f"[SANITY] {label_base}={base_n} | {label_filtered}={filt_n} | delta={base_n - filt_n}")

# Parâmetros didáticos (ajuste durante a aula)
MES_REF = "2021-01-01"        # mês (date) - padrão: primeiro dia do mês
UF_REF = "SP"                # UF para exemplos
DT_REF = "2021-01-02"         # um dia de referência (date) para exemplos
PERIODO_INI = "2021-01-01"    # período para recortes em pedidos
PERIODO_FIM = "2021-02-01"    # exclusivo (até antes dessa data)

# Tabelas (mesmas tabelas da aula anterior)
TBL_GOLD_KPIS = "aulas_ao_vivo.live_20260309.gold_vendas_kpis_mensais"
TBL_GOLD_RECEITA = "aulas_ao_vivo.live_20260309.gold_vendas_receita_mensal_produto_uf"
TBL_SILVER_CAT = "aulas_ao_vivo.live_20260309.silver_catalogo_produtos"
TBL_SILVER_PEDIDOS = "aulas_ao_vivo.live_20260309.silver_vendas_pedidos"

# Criar temp views (ajuda quando o schema/banco muda; mantém SQL simples)
for t in [TBL_GOLD_KPIS, TBL_GOLD_RECEITA, TBL_SILVER_CAT, TBL_SILVER_PEDIDOS]:
    spark.table(t).createOrReplaceTempView(t)


In [0]:
"""
CHECK-IN RÁPIDO — o que existe e como está o schema
O QUE OBSERVAR:
  - nomes de colunas (evita errar SELECT)
  - tipos (date vs timestamp) (evita filtro errado)
"""

print("--- Schema: gold_vendas_kpis_mensais ---")
spark.table(TBL_GOLD_KPIS).printSchema()

print("--- Schema: gold_vendas_receita_mensal_produto_uf ---")
spark.table(TBL_GOLD_RECEITA).printSchema()

print("--- Schema: silver_catalogo_produtos ---")
spark.table(TBL_SILVER_CAT).printSchema()

print("--- Schema: silver_vendas_pedidos ---")
spark.table(TBL_SILVER_PEDIDOS).printSchema()

print("Contagens (para noção de ordem de grandeza):")
for t in [TBL_GOLD_KPIS, TBL_GOLD_RECEITA, TBL_SILVER_CAT, TBL_SILVER_PEDIDOS]:
    print(f"- {t}: {spark.table(t).count()}")


In [0]:
%sql
/*
PERGUNTA 1: Quais foram os TOP 5 meses por receita total?
MÉTRICA: receita_total
POPULAÇÃO: gold_vendas_kpis_mensais (KPIs mensais)
RECORTE: nenhum (apenas ordenação)

VALIDAÇÃO RÁPIDA:
  - conferir se a ordenação faz sentido
  - olhar quantos meses existem no total

Intenção: rankear meses por receita_total
*/

SELECT
  mes_referencia,
  receita_total,
  qtd_pedidos_validos,
  ticket_medio_pedido
FROM aulas_ao_vivo.live_20260309.gold_vendas_kpis_mensais
--WHERE?
ORDER BY 
  receita_total DESC, 
  mes_referencia ASC
LIMIT 5


In [0]:
# Exemplo equivalente em PySpark
"""
Mesma pergunta do Exemplo 1, agora usando DataFrame API.
O que observar: leitura por encadeamento (select -> orderBy -> limit).
"""

k = spark.table(TBL_GOLD_KPIS)

df_q1_py = (
    k.select(
        "mes_referencia",
        "receita_total",
        "qtd_pedidos_validos",
        "ticket_medio_pedido",
    )
    .orderBy(F.col("receita_total").desc(), F.col("mes_referencia").asc())
    .limit(5)
)

d(df_q1_py)


In [0]:
display(spark.sql(
    f"""
SELECT
  mes_referencia,
  receita_total,
  qtd_pedidos_validos,
  ticket_medio_pedido
FROM aulas_ao_vivo.live_20260309.gold_vendas_kpis_mensais
--WHERE '{MES_REF}'
ORDER BY 
  receita_total DESC, 
  mes_referencia ASC
LIMIT 5
    """
))

In [0]:
%sql
/*
PERGUNTA 2: No mês MES_REF e na UF UF_REF, quais são os TOP 10 produtos por receita?
MÉTRICA: receita_total
POPULAÇÃO: gold_vendas_receita_mensal_produto_uf
RECORTE: mes_referencia = MES_REF e uf = UF_REF
APRESENTAÇÃO: ranking (ORDER BY receita_total DESC + desempate por produto)

VALIDAÇÃO RÁPIDA:
  - contar quantos produtos existem nesse recorte
  - garantir que o ranking é determinístico (desempate)


Intenção: top produtos por receita em um mês/UF
*/

SELECT
  mes_referencia,
  uf,
  produto,
  receita_total,
  qtd_pedidos_validos,
  ticket_medio_item
FROM aulas_ao_vivo.live_20260309.gold_vendas_receita_mensal_produto_uf
WHERE mes_referencia = DATE("2021-01-01")
  AND uf = "SP" 
ORDER BY receita_total DESC, produto ASC
LIMIT 10


In [0]:
display(
    spark.sql(
        f"""
SELECT
  mes_referencia,
  uf,
  produto,
  receita_total,
  qtd_pedidos_validos,
  ticket_medio_item
FROM aulas_ao_vivo.live_20260309.gold_vendas_receita_mensal_produto_uf
WHERE mes_referencia = '{MES_REF}'
  AND uf = '{UF_REF}'
ORDER BY receita_total DESC, produto ASC
LIMIT 10
"""
    )
)

In [0]:
%sql
--Validação: quantos produtos existem no recorte (universo da query anterior)

SELECT COUNT(DISTINCT produto) AS qtd_produtos
FROM aulas_ao_vivo.live_20260309.gold_vendas_receita_mensal_produto_uf
WHERE mes_referencia = DATE("2021-01-01" )
  AND uf = "SP" 


In [0]:
# EXEMPLO 2 (PySpark equivalente)
"""
Mesma pergunta do Exemplo 2 em DataFrame API.
O que observar: filtros explícitos + orderBy com desempate.
"""

r = spark.table(TBL_GOLD_RECEITA)

df_q2_base = r.filter(
    (F.col("mes_referencia") == F.to_date(F.lit(MES_REF))) &
    (F.col("uf") == F.lit(UF_REF))
)

df_q2_py = (
    df_q2_base
    .select("mes_referencia", "uf", "produto", "receita_total", "qtd_pedidos_validos", "ticket_medio_item")
    .orderBy(F.col("receita_total").desc(), F.col("produto").asc())
    .limit(10)
)

d(df_q2_py)

print("[VALIDAÇÃO] produtos distintos no recorte:", df_q2_base.select("produto").distinct().count())


In [0]:
# EXEMPLO 2.1 (SQL usando a API do Spark - relacionamento para construir métrica)
"""
ESCALADA: adicionar relacionamento (join) para construir métrica por categoria.

PERGUNTA 2.1: No mês MES_REF (todas as UFs), quais são as TOP 5 categorias por receita?
Tabelas:
  - gold_vendas_receita_mensal_produto_uf (receita por produto/UF/mês)
  - silver_catalogo_produtos (dimensão com categoria e status_produto)

RECORTE:
  - mes_referencia = MES_REF
  - status_produto = 'ativo' (evita categoria com produto inativo)

VALIDAÇÃO RÁPIDA:
  - comparar nº de produtos antes/depois do filtro de status_produto
"""

q21_sql = f"""
SELECT
  c.categoria,
  CAST(SUM(r.receita_total) AS DECIMAL(32,2)) AS receita_total_categoria,
  SUM(r.qtd_pedidos_validos) AS qtd_pedidos_validos
FROM {TBL_GOLD_RECEITA} r
JOIN {TBL_SILVER_CAT} c
  ON r.produto = c.produto
WHERE r.mes_referencia = DATE('{MES_REF}')
  AND c.status_produto = 'ativo'
GROUP BY c.categoria
ORDER BY receita_total_categoria DESC, c.categoria ASC
LIMIT 5
"""

df_q21 = spark.sql(q21_sql)
d(df_q21)

# Validação: cobertura de produtos no join
q21_cov_sql = f"""
SELECT
  COUNT(DISTINCT r.produto) AS produtos_no_gold,
  COUNT(DISTINCT c.produto) AS produtos_no_catalogo,
  COUNT(DISTINCT CASE WHEN c.produto IS NOT NULL THEN r.produto END) AS produtos_com_match
FROM {TBL_GOLD_RECEITA} r
LEFT JOIN {TBL_SILVER_CAT} c
  ON r.produto = c.produto
WHERE r.mes_referencia = DATE('{MES_REF}')
"""
d(spark.sql(q21_cov_sql))


In [0]:
# EXEMPLO 3 (SQL - pedidos cancelados: agregado + amostra)
"""
PERGUNTA 3: No período [PERIODO_INI, PERIODO_FIM), quantos pedidos cancelados por UF?
E também: listar uma amostra dos cancelados mais recentes (para inspeção).

POPULAÇÃO: silver_vendas_pedidos
RECORTE:
  - status = 'cancelado'
  - dt >= PERIODO_INI e dt < PERIODO_FIM

VALIDAÇÃO RÁPIDA:
  - contar universo base no período
  - contar recorte cancelado no período
  - inspecionar linhas recentes (ORDER BY data_pedido DESC)
"""

# 3A) Agregado por UF
q3a_sql = f"""
SELECT
  uf,
  COUNT(DISTINCT id_pedido) AS qtd_pedidos_cancelados,
  CAST(SUM(valor_total_pedido) AS DECIMAL(32,2)) AS valor_total_cancelado
FROM {TBL_SILVER_PEDIDOS}
WHERE status = 'cancelado'
  AND dt >= DATE('{PERIODO_INI}')
  AND dt <  DATE('{PERIODO_FIM}')
GROUP BY uf
ORDER BY qtd_pedidos_cancelados DESC, uf ASC
LIMIT 30
"""

df_q3a = spark.sql(q3a_sql)
d(df_q3a)

# 3B) Amostra dos cancelados mais recentes
q3b_sql = f"""
SELECT
  id_pedido,
  data_pedido,
  dt,
  uf,
  valor_total_pedido,
  quantidade_total,
  qtd_itens_pedido
FROM {TBL_SILVER_PEDIDOS}
WHERE status = 'cancelado'
  AND dt >= DATE('{PERIODO_INI}')
  AND dt <  DATE('{PERIODO_FIM}')
ORDER BY data_pedido DESC, id_pedido ASC
LIMIT 20
"""

df_q3b = spark.sql(q3b_sql)
d(df_q3b)

# Validação: contagem base vs recorte
base_periodo = spark.table(TBL_SILVER_PEDIDOS).filter(
    (F.col("dt") >= F.to_date(F.lit(PERIODO_INI))) &
    (F.col("dt") <  F.to_date(F.lit(PERIODO_FIM)))
)
cancel_periodo = base_periodo.filter(F.col("status") == F.lit("cancelado"))
sanity_counts(base_periodo, cancel_periodo, label_base="PEDIDOS_NO_PERIODO", label_filtered="CANCELADOS_NO_PERIODO")


In [0]:
# EXEMPLO 3 (PySpark equivalente)
"""
Mesma lógica do Exemplo 3 em DataFrame API:
  - agregado por UF
  - amostra por recência
"""

p = spark.table(TBL_SILVER_PEDIDOS)

df_base_periodo = p.filter(
    (F.col("dt") >= F.to_date(F.lit(PERIODO_INI))) &
    (F.col("dt") <  F.to_date(F.lit(PERIODO_FIM)))
)

df_cancel = df_base_periodo.filter(F.col("status") == F.lit("cancelado"))
sanity_counts(df_base_periodo, df_cancel, label_base="PEDIDOS_NO_PERIODO", label_filtered="CANCELADOS_NO_PERIODO")

# Agregado por UF
df_cancel_uf = (
    df_cancel
    .groupBy("uf")
    .agg(
        F.countDistinct("id_pedido").alias("qtd_pedidos_cancelados"),
        F.sum(F.col("valor_total_pedido")).cast(T.DecimalType(32, 2)).alias("valor_total_cancelado")
    )
    .orderBy(F.col("qtd_pedidos_cancelados").desc(), F.col("uf").asc())
    .limit(30)
)
d(df_cancel_uf)

# Amostra recente
df_cancel_sample = (
    df_cancel
    .select("id_pedido", "data_pedido", "dt", "uf", "valor_total_pedido", "quantidade_total", "qtd_itens_pedido")
    .orderBy(F.col("data_pedido").desc(), F.col("id_pedido").asc())
    .limit(20)
)
d(df_cancel_sample)
